# Ensemble Method with Dirichlet Only (No Rotations)

This notebook implements the ensemble clustering method on CIFAR-10 with **ONLY label heterogeneity**:
- **NO feature heterogeneity**: All clients use same transformations (no rotations)
- **Label heterogeneity**: Dirichlet distribution with tunable alpha parameter

**Purpose:**
Isolate the impact of label heterogeneity alone on ensemble clustering performance.

**Three phases:**
1. **Warmup**: All clients train local models with label imbalance only
2. **Clustering**: Group clients using weight differences (FC + Layer4)
3. **Hierarchical Training**: Train cluster-specific feature extractors + shared classifier

**Alpha parameter:**
- `alpha = 0.1` → Highly non-IID (extreme label imbalance per client)
- `alpha = 0.5` → Moderate non-IID
- `alpha = 1.0` → Balanced non-IID
- `alpha = 10.0` → Nearly IID

In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    !pip install -q torch torchvision scikit-learn matplotlib seaborn scipy
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

## Import Libraries

In [ ]:
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.metrics import confusion_matrix, silhouette_score, adjusted_rand_score, classification_report, roc_auc_score, f1_score
from sklearn.preprocessing import label_binarize
from sklearn.cluster import KMeans
import copy
import random
import time
from collections import defaultdict
import gc
import itertools

sys.path.append('..')
from training.utils import get_model, set_seed
from training.ensemble_model import EnsembleModel

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Memory cleanup
torch.cuda.empty_cache() if torch.cuda.is_available() else None
gc.collect()

## Load Configuration

In [ ]:
# Load configuration from JSON
import os
if os.path.exists('config.json'):
    config_path = 'config.json'
elif os.path.exists('experiments/config.json'):
    config_path = 'experiments/config.json'
else:
    raise FileNotFoundError("config.json not found")

with open(config_path, 'r') as f:
    CONFIG = json.load(f)

# Add Dirichlet alpha parameter
CONFIG['dirichlet_alpha'] = 0.5  # ← TUNE THIS: 0.1 (high non-IID) to 10.0 (low non-IID)

# Set random seeds
SEED = CONFIG['seed']
set_seed(SEED)
CONFIG['seed'] = SEED

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

# Memory monitoring
if torch.cuda.is_available():
    print(f"\nGPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB total")
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1e9:.4f} GB")

## Load and Prepare Data

**No rotations applied** - only standard preprocessing and normalization.

In [ ]:
# Standard transform (no rotation)
standard_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10
print("Loading CIFAR-10 dataset...")
train_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
test_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=None)

print(f"Train dataset size: {len(train_dataset_raw)}")
print(f"Test dataset size: {len(test_dataset_raw)}")
print(f"✓ No rotation transformations applied - all clients use same preprocessing")

## Distribute Data with Dirichlet Only

**Only label heterogeneity** - Dirichlet distribution creates non-IID label distributions across clients.

In [ ]:
# Split ratios for train/val only (test uses original CIFAR-10 test set)
train_ratio = 0.8
val_ratio = 0.2

print(f"Data split ratios: Train={train_ratio}, Val={val_ratio}")
print(f"Test uses original CIFAR-10 test set (10,000 samples)")

# Organize data by class
num_classes = 10
indices_by_class = [[] for _ in range(num_classes)]

for idx, (_, label) in enumerate(train_dataset_raw):
    indices_by_class[label].append(idx)

print(f"\nTotal samples per class:")
for class_id in range(num_classes):
    print(f"  Class {class_id}: {len(indices_by_class[class_id])} samples")

# Split each class into train/val only (NO test split)
train_indices_by_class = [[] for _ in range(num_classes)]
val_indices_by_class = [[] for _ in range(num_classes)]

for class_id in range(num_classes):
    class_indices = np.array(indices_by_class[class_id])
    np.random.shuffle(class_indices)
    
    n = len(class_indices)
    train_end = int(n * train_ratio)
    
    train_indices_by_class[class_id] = class_indices[:train_end].tolist()
    val_indices_by_class[class_id] = class_indices[train_end:].tolist()

print(f"\nSplit samples per class:")
for class_id in range(num_classes):
    print(f"  Class {class_id}: Train={len(train_indices_by_class[class_id])}, Val={len(val_indices_by_class[class_id])}")

# Apply Dirichlet distribution to train, IID to val
print(f"\nApplying Dirichlet distribution (alpha={CONFIG['dirichlet_alpha']}) to train...")
print(f"Validation will be IID (uniform distribution)")
print(f"Test uses original CIFAR-10 test set (separate)")

def distribute_with_dirichlet(indices_by_class, num_clients, alpha):
    \"\"\"Distribute data to clients using Dirichlet distribution.\"\"\"
    client_indices = [[] for _ in range(num_clients)]
    
    for class_id in range(len(indices_by_class)):
        class_indices = np.array(indices_by_class[class_id])
        np.random.shuffle(class_indices)
        
        # Sample proportions from Dirichlet distribution
        proportions = np.random.dirichlet(alpha=[alpha] * num_clients)
        proportions = (np.cumsum(proportions) * len(class_indices)).astype(int)[:-1]
        
        # Split indices according to proportions
        splits = np.split(class_indices, proportions)
        
        for client_idx, split in enumerate(splits):
            client_indices[client_idx].extend(split.tolist())
    
    # Shuffle each client's indices
    for client_idx in range(num_clients):
        random.shuffle(client_indices[client_idx])
    
    return client_indices

def distribute_iid(indices_by_class, num_clients):
    \"\"\"Distribute data uniformly (IID) across clients.\"\"\"
    # Flatten all indices
    all_indices = []
    for class_indices in indices_by_class:
        all_indices.extend(class_indices)
    
    # Shuffle
    random.shuffle(all_indices)
    
    # Split uniformly
    client_indices = [[] for _ in range(num_clients)]
    samples_per_client = len(all_indices) // num_clients
    
    for client_idx in range(num_clients):
        start = client_idx * samples_per_client
        end = start + samples_per_client if client_idx < num_clients - 1 else len(all_indices)
        client_indices[client_idx] = all_indices[start:end]
    
    return client_indices

# Apply Dirichlet to train, IID to val (NO test distribution)
client_indices_train = distribute_with_dirichlet(train_indices_by_class, CONFIG['num_clients'], CONFIG['dirichlet_alpha'])
client_indices_val = distribute_iid(val_indices_by_class, CONFIG['num_clients'])

# Create datasets for each client (no test subsets)
class StandardCIFAR10Dataset(Dataset):
    \"\"\"CIFAR-10 dataset with standard preprocessing only.\"\"\"
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        image = self.transform(image)
        return image, label

# Create standard dataset
standard_dataset = StandardCIFAR10Dataset(train_dataset_raw, standard_transform)

train_subsets = []
val_subsets = []
client_label_distributions = []

print(f"\nCreating datasets with standard preprocessing (no rotations)...")

for client_idx in range(CONFIG['num_clients']):
    # Create train and val subsets only
    train_subset = Subset(standard_dataset, client_indices_train[client_idx])
    val_subset = Subset(standard_dataset, client_indices_val[client_idx])
    
    train_subsets.append(train_subset)
    val_subsets.append(val_subset)
    
    # Track label distribution for this client (using train data)
    labels = [train_dataset_raw[idx][1] for idx in client_indices_train[client_idx]]
    label_dist = np.bincount(labels, minlength=num_classes)
    client_label_distributions.append(label_dist)

print(f"\nCreated {len(train_subsets)} client train datasets")
print(f"Created {len(val_subsets)} client validation datasets")
print(f"Note: Test dataset uses original CIFAR-10 test set (10,000 samples)")
print(f"\nAverage samples per client:")
print(f"  Train: {np.mean([len(s) for s in train_subsets]):.1f}")
print(f"  Validation: {np.mean([len(s) for s in val_subsets]):.1f}")
print(f"\nTotal samples:")
print(f"  Train: {sum([len(s) for s in train_subsets])}")
print(f"  Validation: {sum([len(s) for s in val_subsets])}")
print(f"  Note: Test set (10,000) is separate (original CIFAR-10)")
print(f"\n✓ Train: Dirichlet (non-IID, alpha={CONFIG['dirichlet_alpha']})")
print(f"✓ Val: IID (uniform distribution)")
print(f"✓ Test: Original CIFAR-10 test set (not split among clients)")


## Visualize Label Distribution

In [ ]:
# Visualize label distribution heterogeneity
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Samples per client
ax = axes[0, 0]
client_sizes = [len(subset) for subset in train_subsets]
ax.bar(range(CONFIG['num_clients']), client_sizes, alpha=0.7)
ax.axhline(y=np.mean(client_sizes), color='red', linestyle='--', label=f'Mean: {np.mean(client_sizes):.0f}')
ax.set_xlabel('Client ID')
ax.set_ylabel('Number of Samples')
ax.set_title(f'Data Distribution Across Clients\n(Std: {np.std(client_sizes):.1f})')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Label distribution heatmap
ax = axes[0, 1]
label_dist_matrix = np.array(client_label_distributions).T
im = ax.imshow(label_dist_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xlabel('Client ID')
ax.set_ylabel('Class Label')
ax.set_title(f'Label Distribution per Client\n(Dirichlet α={CONFIG["dirichlet_alpha"]})')
plt.colorbar(im, ax=ax, label='Sample Count')

# Plot 3: Classes per client histogram
ax = axes[1, 0]
classes_per_client = [np.sum(dist > 0) for dist in client_label_distributions]
ax.hist(classes_per_client, bins=range(1, 12), alpha=0.7, edgecolor='black')
ax.set_xlabel('Number of Classes Present')
ax.set_ylabel('Number of Clients')
ax.set_title(f'Classes per Client Distribution\n(Mean: {np.mean(classes_per_client):.2f})')
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: Label entropy per client (measure of balance)
ax = axes[1, 1]
entropies = []
for dist in client_label_distributions:
    probs = dist / dist.sum()
    probs = probs[probs > 0]  # Remove zeros
    entropy = -np.sum(probs * np.log2(probs))
    entropies.append(entropy)

ax.hist(entropies, bins=20, alpha=0.7, edgecolor='black')
ax.axvline(x=np.log2(num_classes), color='red', linestyle='--', label=f'Max (uniform): {np.log2(num_classes):.2f}')
ax.set_xlabel('Entropy (bits)')
ax.set_ylabel('Number of Clients')
ax.set_title(f'Label Distribution Entropy\n(Mean: {np.mean(entropies):.2f}, Lower = more imbalanced)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ensemble_dirichlet_only_data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nLabel distribution statistics:")
print(f"  Mean entropy: {np.mean(entropies):.3f} bits")
print(f"  Max possible entropy: {np.log2(num_classes):.3f} bits (uniform)")
print(f"  Entropy std: {np.std(entropies):.3f}")

## Create Validation and Test Datasets

In [ ]:
# Create validation dataset (combine all client validation sets)
print("Creating validation dataset...")
val_dataset = torch.utils.data.ConcatDataset(val_subsets)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Validation dataset size: {len(val_dataset)}")

# Create test dataset using the original CIFAR-10 test set (NOT split among clients)
print("Creating test dataset...")
test_dataset_full = torchvision.datasets.CIFAR10(
    root='./data', 
    train=False, 
    download=True, 
    transform=standard_transform  # Use same normalization as training
)
test_loader = DataLoader(test_dataset_full, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Test dataset size: {len(test_dataset_full)}")
print(f"\n✓ Validation: Combined from all clients (IID distribution)")
print(f"✓ Test: Original CIFAR-10 test set (10,000 samples)")
print(f"⚠️  Test set will ONLY be evaluated at the end (proper ML practice)")

## Phase 1: Warmup Training

Each client trains a local model on their label distribution for a few epochs.

In [ ]:
def train_local_model(model, train_loader, epochs, lr):
    """Train local model for specified epochs."""
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    return model

print("Warmup training function defined")

In [ ]:
# Warmup configuration
warmup_epochs = CONFIG['warmup_epochs']

print(f"{'='*70}")
print(f"PHASE 1: WARMUP TRAINING (Dirichlet α={CONFIG['dirichlet_alpha']}, NO ROTATIONS)")
print(f"{'='*70}")
print(f"Number of clients: {CONFIG['num_clients']}")
print(f"Warmup epochs: {warmup_epochs}")
print(f"{'='*70}\n")

client_models = []
warmup_start = time.time()

for client_idx in range(CONFIG['num_clients']):
    # Create local model
    local_model = get_model(
        model_name=CONFIG['model_name'],
        num_classes=10,
        pretrained=CONFIG['pretrained']
    ).to(device)
    
    # Create data loader
    train_loader = DataLoader(
        train_subsets[client_idx],
        batch_size=CONFIG['batch_size'],
        shuffle=True
    )
    
    # Train locally
    local_model = train_local_model(local_model, train_loader, warmup_epochs, CONFIG['lr'])
    
    # Store model
    client_models.append(local_model)
    
    if (client_idx + 1) % 10 == 0:
        print(f"Completed warmup for {client_idx + 1}/{CONFIG['num_clients']} clients")

warmup_time = time.time() - warmup_start

print(f"\n{'='*70}")
print(f"Warmup Complete!")
print(f"{'='*70}")
print(f"Time: {warmup_time:.2f}s ({warmup_time/60:.2f} min)")
print(f"Avg time per client: {warmup_time/CONFIG['num_clients']:.2f}s")

# Memory cleanup after warmup
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

if torch.cuda.is_available():
    print(f"GPU Memory allocated after warmup: {torch.cuda.memory_allocated() / 1e9:.4f} GB")

## Phase 2: Extract Weights and Cluster Clients

Extract FC and Layer4 weights from all clients and cluster them using K-Means.

**Question: Without rotation-based feature heterogeneity, will clustering find meaningful patterns based on label distributions alone?**

In [ ]:
def extract_fc_layer4_weights(model):
    """Extract FC layer and Layer4 weights as flattened vector."""
    weight_vector = []
    
    state_dict = model.state_dict()
    
    # Extract Layer4 weights (ResNet18)
    for key in state_dict.keys():
        if 'layer4' in key and ('weight' in key or 'bias' in key):
            weight_vector.append(state_dict[key].flatten().cpu().numpy())
    
    # Extract FC layer weights
    if 'fc.weight' in state_dict:
        weight_vector.append(state_dict['fc.weight'].flatten().cpu().numpy())
    if 'fc.bias' in state_dict:
        weight_vector.append(state_dict['fc.bias'].flatten().cpu().numpy())
    
    return np.concatenate(weight_vector)

print("Weight extraction function defined")

In [ ]:
print(f"{'='*70}")
print("PHASE 2: CLUSTERING (Label Heterogeneity Only)")
print(f"{'='*70}")

# Extract weights from all client models
print("Extracting weights from all clients...")
client_weight_vectors = []

for client_idx, model in enumerate(client_models):
    weight_vec = extract_fc_layer4_weights(model)
    client_weight_vectors.append(weight_vec)

client_weight_matrix = np.array(client_weight_vectors)
print(f"Weight matrix shape: {client_weight_matrix.shape}")

# Compute weight differences from mean
mean_weights = client_weight_matrix.mean(axis=0)
weight_differences = client_weight_matrix - mean_weights

# Normalize
weight_differences = weight_differences / (np.linalg.norm(weight_differences, axis=1, keepdims=True) + 1e-8)

print(f"Normalized weight differences shape: {weight_differences.shape}")

# K-Means clustering
num_clusters = CONFIG['num_clusters_model']
print(f"\nApplying K-Means clustering (K={num_clusters})...")

kmeans = KMeans(n_clusters=num_clusters, random_state=SEED, n_init=10)
cluster_labels = kmeans.fit_predict(weight_differences)

# Cluster statistics
print(f"\nCluster assignment:")
for cluster_id in range(num_clusters):
    members = np.where(cluster_labels == cluster_id)[0]
    print(f"  Cluster {cluster_id}: {len(members)} clients")
    
    # Show label distribution statistics in this cluster
    cluster_dists = [client_label_distributions[i] for i in members]
    avg_entropy = np.mean([entropies[i] for i in members])
    print(f"    Average label entropy: {avg_entropy:.3f} bits")

# Clustering quality metrics
silhouette_avg = silhouette_score(weight_differences, cluster_labels)
print(f"\nSilhouette score: {silhouette_avg:.4f}")
print(f"(Higher score = better clustering, based on label heterogeneity patterns)")

print(f"{'='*70}\n")

# Memory cleanup
del client_weight_vectors, client_weight_matrix, weight_differences
gc.collect()

## Phase 3: Hierarchical Training

Train cluster-specific feature extractors with a shared classifier.

In [ ]:
# Create ensemble model
print(f"{'='*70}")
print("PHASE 3: HIERARCHICAL TRAINING (Pooled Data)")
print(f"{'='*70}")

# Import ResNetFeatureExtractor
from training.ensemble_model import ResNetFeatureExtractor

# Create feature extractors for each cluster from cluster representatives
feature_extractors = []

print("\nInitializing feature extractors from cluster representatives...")
for cluster_id in range(num_clusters):
    cluster_members = np.where(cluster_labels == cluster_id)[0]
    
    # Use first member as representative
    representative_idx = cluster_members[0]
    representative_model = client_models[representative_idx]
    
    # Create feature extractor from representative model (removes FC layer)
    feature_extractor = ResNetFeatureExtractor(representative_model)
    feature_extractors.append(feature_extractor)
    
    print(f"  Cluster {cluster_id}: Initialized from client {representative_idx} ({len(cluster_members)} members)")

# Create ensemble model with the feature extractors
ensemble_model = EnsembleModel(
    models=feature_extractors,
    feature_dim=512,  # ResNet18 output
    num_classes=10
).to(device)

print("\nEnsemble model created with cluster-specific feature extractors")

# Memory cleanup - delete client models as we don't need them anymore
del client_models
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# Hierarchical training configuration
training_rounds = CONFIG['training_rounds']
local_epochs = CONFIG['local_epochs']
optimizer = optim.SGD(ensemble_model.parameters(), lr=CONFIG['lr'], momentum=0.9)
criterion = nn.CrossEntropyLoss()

print(f"Hierarchical training rounds: {training_rounds}")
print(f"Local epochs per cluster: {local_epochs}")
print(f"Learning rate: {CONFIG['lr']}")

# Calculate total training budget
warmup_budget = warmup_epochs * CONFIG['num_clients']
hierarchical_budget = training_rounds * CONFIG['num_clients'] * local_epochs
total_budget = warmup_budget + hierarchical_budget
print(f"\nEnsemble Training Budget:")
print(f"  Warmup: {warmup_epochs} epochs × {CONFIG['num_clients']} clients = {warmup_budget} client-epochs")
print(f"  Hierarchical: {training_rounds} rounds × {CONFIG['num_clients']} clients × {local_epochs} epochs = {hierarchical_budget} client-epochs")
print(f"  Total: {total_budget} client-epochs")
print(f"{'='*70}\n")

# Storage for metrics
val_losses = []
val_accs = []
round_times = []

# NEW: Storage for per-cluster metrics
cluster_val_losses = defaultdict(list)
cluster_val_accs = defaultdict(list)

# Training loop
training_start = time.time()

for round_num in range(1, training_rounds + 1):
    round_start = time.time()
    ensemble_model.train()
    
    # Train on all clusters
    for cluster_id in range(num_clusters):
        cluster_members = np.where(cluster_labels == cluster_id)[0]
        
        # Combine data from all cluster members (POOLED - not federated)
        cluster_dataset = torch.utils.data.ConcatDataset([
            train_subsets[client_idx] for client_idx in cluster_members
        ])
        
        cluster_loader = DataLoader(
            cluster_dataset,
            batch_size=CONFIG['batch_size'],
            shuffle=True
        )
        
        # Train for local_epochs on this cluster
        for epoch in range(local_epochs):
            for inputs, labels in cluster_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                optimizer.zero_grad()
                outputs = ensemble_model(inputs, [cluster_id])
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
    
    # Evaluate on VALIDATION set (not test!)
    ensemble_model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    # Track per-cluster metrics for this round
    cluster_val_losses_epoch = defaultdict(float)
    cluster_correct_epoch = defaultdict(int)
    cluster_total_epoch = defaultdict(int)
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Average predictions from all feature extractors
            batch_predictions = []
            for cluster_id in range(num_clusters):
                outputs = ensemble_model(inputs, [cluster_id])
                batch_predictions.append(outputs)
                
                # Track per-cluster performance
                loss_c = criterion(outputs, labels)
                cluster_val_losses_epoch[cluster_id] += loss_c.item() * inputs.size(0)
                _, predicted_c = outputs.max(1)
                cluster_correct_epoch[cluster_id] += predicted_c.eq(labels).sum().item()
                cluster_total_epoch[cluster_id] += labels.size(0)
            
            # Average logits (Ensemble performance)
            avg_outputs = torch.stack(batch_predictions).mean(dim=0)
            loss = criterion(avg_outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = avg_outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    # Calculate and store ensemble metrics
    avg_val_loss = val_loss / total
    val_acc = correct / total
    
    val_losses.append(avg_val_loss)
    val_accs.append(val_acc)
    
    # Calculate and store per-cluster metrics
    for cluster_id in range(num_clusters):
        if cluster_total_epoch[cluster_id] > 0:
            c_loss = cluster_val_losses_epoch[cluster_id] / cluster_total_epoch[cluster_id]
            c_acc = cluster_correct_epoch[cluster_id] / cluster_total_epoch[cluster_id]
            cluster_val_losses[cluster_id].append(c_loss)
            cluster_val_accs[cluster_id].append(c_acc)
        else:
            # Should not happen with current val loader Setup
            cluster_val_losses[cluster_id].append(0.0)
            cluster_val_accs[cluster_id].append(0.0)
    
    round_time = time.time() - round_start
    round_times.append(round_time)
    
    # Print progress
    if round_num % 5 == 0 or round_num == 1:
        print(f"Round {round_num}/{training_rounds} - "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}, "
              f"Time: {round_time:.2f}s")
        # Print cluster performance summary
        cluster_acc_str = ", ".join([f"C{c}: {cluster_val_accs[c][-1]:.3f}" for c in range(num_clusters)])
        print(f"  Cluster Accuracies: {cluster_acc_str}")

training_time = time.time() - training_start
total_time = warmup_time + training_time

print(f"\n{'='*70}")
print(f"Hierarchical Training Complete!")
print(f"{'='*70}")
print(f"Training time: {training_time:.2f}s ({training_time/60:.2f} min)")
print(f"Total time (warmup + training): {total_time:.2f}s ({total_time/60:.2f} min)")
print(f"Average time per round: {np.mean(round_times):.2f}s")
print(f"Final validation accuracy: {val_accs[-1]:.4f}")
print(f"Best validation accuracy: {max(val_accs):.4f} (round {np.argmax(val_accs)+1})")


## Final Test Set Evaluation

In [ ]:
# Final evaluation on TEST set (only once!)
print(f"{'='*70}")
print("FINAL TEST SET EVALUATION")
print(f"{'='*70}\n")

ensemble_model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Average predictions from all feature extractors
        batch_predictions = []
        for cluster_id in range(num_clusters):
            outputs = ensemble_model(inputs, [cluster_id])
            batch_predictions.append(outputs)
        
        # Average logits
        avg_outputs = torch.stack(batch_predictions).mean(dim=0)
        loss = criterion(avg_outputs, labels)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = avg_outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

final_test_loss = test_loss / total
final_test_acc = correct / total

print(f"Final Test Loss: {final_test_loss:.4f}")
print(f"Final Test Accuracy: {final_test_acc:.4f}")
print(f"\nComparison:")
print(f"  Best Validation Accuracy: {max(val_accs):.4f} (round {np.argmax(val_accs)+1})")
print(f"  Final Test Accuracy: {final_test_acc:.4f}")
print(f"  Difference (Val - Test): {max(val_accs) - final_test_acc:+.4f}")
print(f"{'='*70}\n")

In [ ]:
# Collect predictions and labels on test set
print("Collecting predictions on test set...")
ensemble_model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Average predictions from all feature extractors
        batch_predictions = []
        for cluster_id in range(num_clusters):
            outputs = ensemble_model(inputs, [cluster_id])
            batch_predictions.append(outputs)
        
        # Average logits
        avg_outputs = torch.stack(batch_predictions).mean(dim=0)
        
        # Get probabilities for AUC
        probs = torch.nn.functional.softmax(avg_outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        
        _, predicted = avg_outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_probs = np.concatenate(all_probs, axis=0)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"\n{'='*70}")
print("DETAILED TEST SET METRICS")
print(f"{'='*70}\n")

# 1. Classification Report
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# 2. Overall Metrics
macro_f1 = f1_score(all_labels, all_preds, average='macro')
micro_f1 = f1_score(all_labels, all_preds, average='micro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"\nF1-Score Summary:")
print(f"  Macro F1:    {macro_f1:.4f}")
print(f"  Micro F1:    {micro_f1:.4f}")
print(f"  Weighted F1: {weighted_f1:.4f}")

# 3. ROC-AUC Score
try:
    labels_binarized = label_binarize(all_labels, classes=range(10))
    macro_auc = roc_auc_score(labels_binarized, all_probs, average='macro')
    weighted_auc = roc_auc_score(labels_binarized, all_probs, average='weighted')
    per_class_auc = roc_auc_score(labels_binarized, all_probs, average=None)
    
    print(f"\nROC-AUC Summary:")
    print(f"  Macro AUC:    {macro_auc:.4f}")
    print(f"  Weighted AUC: {weighted_auc:.4f}")
    print(f"\nPer-class AUC:")
    for i, name in enumerate(class_names):
        print(f"  {name:12s}: {per_class_auc[i]:.4f}")
except Exception as e:
    print(f"\nNote: Could not compute AUC scores: {e}")

print(f"\n{'='*70}\n")

## Confusion Matrix Visualization

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Counts
ax = axes[0]
im1 = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im1, ax=ax)
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       xlabel='Predicted Label',
       ylabel='True Label',
       title=f'Confusion Matrix - Counts\nEnsemble Dirichlet Only (α={CONFIG["dirichlet_alpha"]})')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, format(cm[i, j], 'd'),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=9)

# Plot 2: Percentages
ax = axes[1]
im2 = ax.imshow(cm_percent, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im2, ax=ax, format='%.1f%%')
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       xlabel='Predicted Label',
       ylabel='True Label',
       title=f'Confusion Matrix - Percentages\nEnsemble Dirichlet Only (α={CONFIG["dirichlet_alpha"]})')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = cm_percent.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, format(cm_percent[i, j], '.1f'),
            ha="center", va="center",
            color="white" if cm_percent[i, j] > thresh else "black",
            fontsize=9)

plt.tight_layout()
plt.savefig(f'ensemble_dirichlet_only_alpha{CONFIG["dirichlet_alpha"]}_confusion_matrix.png', 
            dpi=300, bbox_inches='tight')
plt.show()

print(f"Confusion matrix saved")

# Per-class accuracy
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
print(f"\nPer-class Accuracy:")
for i, name in enumerate(class_names):
    print(f"  {name:12s}: {per_class_accuracy[i]:.4f} ({per_class_accuracy[i]*100:.2f}%)")

## Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Ensemble Validation loss
ax = axes[0, 0]
rounds_range = range(1, training_rounds + 1)
ax.plot(rounds_range, val_losses, 'o-', linewidth=2, markersize=4, label='Ensemble')
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Validation Loss', fontsize=12)
ax.set_title(f'Ensemble Validation Loss (α={CONFIG["dirichlet_alpha"]})', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 2: Ensemble Validation accuracy
ax = axes[0, 1]
ax.plot(rounds_range, val_accs, 's-', linewidth=2, markersize=4, color='green', label='Ensemble')
ax.axhline(y=max(val_accs), color='red', linestyle='--', alpha=0.5, 
           label=f'Best Val: {max(val_accs):.4f}')
ax.axhline(y=final_test_acc, color='blue', linestyle='--', alpha=0.5,
           label=f'Final Test: {final_test_acc:.4f}')
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title(f'Ensemble Validation Accuracy (α={CONFIG["dirichlet_alpha"]})', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

# Plot 3: Cluster Validation Losses (NEW with BOTH)
ax = axes[1, 0]
for cluster_id in range(num_clusters):
    ax.plot(rounds_range, cluster_val_losses[cluster_id], linestyle='-', alpha=0.7, label=f'Cluster {cluster_id}')
ax.plot(rounds_range, val_losses, 'k--', linewidth=2, label='Ensemble (Avg)', alpha=0.8)
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Validation Loss', fontsize=12)
ax.set_title(f'Cluster vs Ensemble Loss', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Cluster Validation Accuracies (NEW with BOTH)
ax = axes[1, 1]
for cluster_id in range(num_clusters):
    ax.plot(rounds_range, cluster_val_accs[cluster_id], linestyle='-', alpha=0.7, label=f'Cluster {cluster_id}')
ax.plot(rounds_range, val_accs, 'k--', linewidth=2, label='Ensemble (Avg)', alpha=0.8)
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title(f'Cluster vs Ensemble Accuracy', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(f'ensemble_dirichlet_only_alpha{CONFIG["dirichlet_alpha"]}_curves_detailed.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Detailed training curves saved")


## Cluster Combination Analysis

Evaluate the performance of different cluster combinations to understand:
- **How many clusters are needed?** (e.g., 2 clusters vs 3 clusters)
- **Which cluster combinations work best?** (e.g., Cluster 0+1 vs Cluster 0+2)
- **Cluster complementarity**: Do specific clusters together perform better?

In [ ]:
def evaluate_cluster_combination(cluster_ids, data_loader, ensemble_model, criterion, desc=""):
    """Evaluate ensemble using only specified clusters."""
    ensemble_model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Get predictions from specified clusters only
            batch_predictions = []
            for cluster_id in cluster_ids:
                outputs = ensemble_model(inputs, [cluster_id])
                batch_predictions.append(outputs)
            
            # Average logits from selected clusters
            avg_outputs = torch.stack(batch_predictions).mean(dim=0)
            loss = criterion(avg_outputs, labels)
            
            total_loss += loss.item() * inputs.size(0)
            _, predicted = avg_outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_loss = total_loss / total
    accuracy = correct / total
    
    return avg_loss, accuracy

print("Cluster combination evaluation function defined")

In [ ]:
print(f"{'='*70}")
print("CLUSTER COMBINATION ANALYSIS")
print(f"{'='*70}\n")

# Storage for combination results
combination_results = {}

# 1. Individual clusters
print("\n1. INDIVIDUAL CLUSTER PERFORMANCE")
print("-" * 50)
for cluster_id in range(num_clusters):
    test_loss, test_acc = evaluate_cluster_combination(
        [cluster_id], test_loader, ensemble_model, criterion
    )
    combination_results[f"cluster_{cluster_id}"] = {
        'clusters': [cluster_id],
        'test_loss': test_loss,
        'test_acc': test_acc,
        'num_clients': int(np.sum(cluster_labels == cluster_id))
    }
    print(f"  Cluster {cluster_id} only: Acc = {test_acc:.4f} ({test_acc*100:.2f}%), "
          f"Loss = {test_loss:.4f} ({combination_results[f'cluster_{cluster_id}']['num_clients']} clients)")

# 2. Pairwise combinations (if we have 3+ clusters)
if num_clusters >= 3:
    print("\n2. PAIRWISE CLUSTER COMBINATIONS")
    print("-" * 50)
    from itertools import combinations
    
    for combo in combinations(range(num_clusters), 2):
        combo_list = list(combo)
        test_loss, test_acc = evaluate_cluster_combination(
            combo_list, test_loader, ensemble_model, criterion
        )
        combo_key = f"clusters_{'_'.join(map(str, combo_list))}"
        combination_results[combo_key] = {
            'clusters': combo_list,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'num_clients': int(sum(np.sum(cluster_labels == c) for c in combo_list))
        }
        print(f"  Clusters {combo_list}: Acc = {test_acc:.4f} ({test_acc*100:.2f}%), "
              f"Loss = {test_loss:.4f} ({combination_results[combo_key]['num_clients']} clients)")

# 3. Triple combinations (if we have 4+ clusters)
if num_clusters >= 4:
    print("\n3. TRIPLE CLUSTER COMBINATIONS")
    print("-" * 50)
    
    for combo in combinations(range(num_clusters), 3):
        combo_list = list(combo)
        test_loss, test_acc = evaluate_cluster_combination(
            combo_list, test_loader, ensemble_model, criterion
        )
        combo_key = f"clusters_{'_'.join(map(str, combo_list))}"
        combination_results[combo_key] = {
            'clusters': combo_list,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'num_clients': int(sum(np.sum(cluster_labels == c) for c in combo_list))
        }
        print(f"  Clusters {combo_list}: Acc = {test_acc:.4f} ({test_acc*100:.2f}%), "
              f"Loss = {test_loss:.4f} ({combination_results[combo_key]['num_clients']} clients)")

# 4. All clusters (full ensemble)
print("\n4. FULL ENSEMBLE (All Clusters)")
print("-" * 50)
all_clusters = list(range(num_clusters))
combination_results['all_clusters'] = {
    'clusters': all_clusters,
    'test_loss': float(final_test_loss),
    'test_acc': float(final_test_acc),
    'num_clients': CONFIG['num_clients']
}
print(f"  All {num_clusters} clusters: Acc = {final_test_acc:.4f} ({final_test_acc*100:.2f}%), "
      f"Loss = {final_test_loss:.4f}")

# 5. Find best combinations
print(f"\n{'='*70}")
print("BEST COMBINATIONS SUMMARY")
print(f"{'='*70}")

# Sort by accuracy
sorted_combos = sorted(combination_results.items(), key=lambda x: x[1]['test_acc'], reverse=True)

print("\nTop 5 Cluster Combinations by Test Accuracy:")
for i, (combo_name, combo_data) in enumerate(sorted_combos[:5], 1):
    cluster_str = '+'.join(map(str, combo_data['clusters']))
    print(f"  {i}. Clusters [{cluster_str}]: "
          f"Acc = {combo_data['test_acc']:.4f} ({combo_data['test_acc']*100:.2f}%), "
          f"Loss = {combo_data['test_loss']:.4f}")

# Group by number of clusters
print("\n\nBest Combination by Number of Clusters:")
best_by_size = {}
for combo_name, combo_data in combination_results.items():
    size = len(combo_data['clusters'])
    if size not in best_by_size or combo_data['test_acc'] > best_by_size[size]['test_acc']:
        best_by_size[size] = combo_data

for size in sorted(best_by_size.keys()):
    combo_data = best_by_size[size]
    cluster_str = '+'.join(map(str, combo_data['clusters']))
    print(f"  {size} cluster(s): [{cluster_str}] → "
          f"Acc = {combo_data['test_acc']:.4f} ({combo_data['test_acc']*100:.2f}%)")

print(f"\n{'='*70}\n")

In [ ]:
# Visualize cluster combination performance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Accuracy by number of clusters
ax = axes[0]
sizes = []
accs = []
for combo_name, combo_data in combination_results.items():
    sizes.append(len(combo_data['clusters']))
    accs.append(combo_data['test_acc'])

# Scatter plot
ax.scatter(sizes, accs, alpha=0.6, s=100)

# Add best-fit trend line
unique_sizes = sorted(set(sizes))
avg_accs = [np.mean([accs[i] for i in range(len(sizes)) if sizes[i] == s]) for s in unique_sizes]
ax.plot(unique_sizes, avg_accs, 'r--', linewidth=2, alpha=0.7, label='Average')

ax.set_xlabel('Number of Clusters Used', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Test Accuracy vs Number of Clusters', fontsize=14, fontweight='bold')
ax.set_xticks(unique_sizes)
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3)
ax.legend()

# Plot 2: Top combinations comparison
ax = axes[1]
top_n = min(10, len(sorted_combos))
top_combos = sorted_combos[:top_n]
combo_names = ['+'.join(map(str, combo_data['clusters'])) for _, combo_data in top_combos]
combo_accs = [combo_data['test_acc'] for _, combo_data in top_combos]

# Different colors by number of clusters
colors = []
for _, combo_data in top_combos:
    n = len(combo_data['clusters'])
    if n == 1:
        colors.append('lightcoral')
    elif n == 2:
        colors.append('lightblue')
    elif n == 3:
        colors.append('lightgreen')
    else:
        colors.append('gold')

bars = ax.barh(range(top_n), combo_accs, color=colors, alpha=0.7, edgecolor='black')
ax.set_yticks(range(top_n))
ax.set_yticklabels(combo_names)
ax.set_xlabel('Test Accuracy', fontsize=12)
ax.set_ylabel('Cluster Combination', fontsize=12)
ax.set_title(f'Top {top_n} Cluster Combinations', fontsize=14, fontweight='bold')
ax.set_xlim([0, 1])
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()

# Add value labels on bars
for i, (bar, acc) in enumerate(zip(bars, combo_accs)):
    ax.text(acc + 0.01, i, f'{acc:.4f}', va='center', fontsize=9)

# Add legend for colors
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lightcoral', edgecolor='black', alpha=0.7, label='1 cluster'),
    Patch(facecolor='lightblue', edgecolor='black', alpha=0.7, label='2 clusters'),
    Patch(facecolor='lightgreen', edgecolor='black', alpha=0.7, label='3 clusters'),
    Patch(facecolor='gold', edgecolor='black', alpha=0.7, label='All clusters')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(f'ensemble_dirichlet_only_alpha{CONFIG["dirichlet_alpha"]}_cluster_combinations.png', 
            dpi=300, bbox_inches='tight')
plt.show()

print("Cluster combination analysis plot saved")

## Save Results

In [ ]:
# Save results
results = {
    'method': 'ensemble_dirichlet_only',
    'config': CONFIG,
    'heterogeneity': 'label_only',
    'warmup_epochs': warmup_epochs,
    'training_rounds': training_rounds,
    'num_clusters': num_clusters,
    'dirichlet_alpha': CONFIG['dirichlet_alpha'],
    'times': {
        'warmup_time': warmup_time,
        'training_time': training_time,
        'total_time': total_time,
        'avg_round_time': float(np.mean(round_times))
    },
    'performance': {
        'best_val_acc': float(max(val_accs)),
        'best_val_round': int(np.argmax(val_accs) + 1),
        'final_val_acc': float(val_accs[-1]),
        'final_test_acc': float(final_test_acc),
        'final_test_loss': float(final_test_loss),
        'val_test_gap': float(max(val_accs) - final_test_acc)
    },
    'val_losses': [float(x) for x in val_losses],
    'val_accs': [float(x) for x in val_accs],
    # NEW: Save per-cluster metrics
    'cluster_performance': {
        'val_losses': {int(k): [float(x) for x in v] for k, v in cluster_val_losses.items()},
        'val_accs': {int(k): [float(x) for x in v] for k, v in cluster_val_accs.items()}
    },
    'test_metrics': {
        'macro_f1': float(macro_f1),
        'micro_f1': float(micro_f1),
        'weighted_f1': float(weighted_f1),
        'macro_auc': float(macro_auc) if 'macro_auc' in locals() else None,
        'weighted_auc': float(weighted_auc) if 'weighted_auc' in locals() else None,
        'per_class_auc': [float(x) for x in per_class_auc] if 'per_class_auc' in locals() else None,
        'per_class_accuracy': [float(x) for x in per_class_accuracy],
        'confusion_matrix': cm.tolist()
    },
    'clustering': {
        'silhouette_score': float(silhouette_avg),
        'cluster_sizes': [int(np.sum(cluster_labels == i)) for i in range(num_clusters)]
    },
    'combination_results': {k: v for k, v in combination_results.items()} if 'combination_results' in locals() else {},
    },
    'data_stats': {
        'mean_train_samples_per_client': float(np.mean([len(s) for s in train_subsets])),
        'mean_val_samples_per_client': float(np.mean([len(s) for s in val_subsets])),
        'total_train_samples': sum([len(s) for s in train_subsets]),
        'total_val_samples': sum([len(s) for s in val_subsets]),
        'mean_label_entropy': float(np.mean(entropies)),
        'std_label_entropy': float(np.std(entropies))
    }
}

filename = f'ensemble_dirichlet_only_alpha{CONFIG["dirichlet_alpha"]}_results.json'
with open(filename, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to '{filename}'")

# Save model checkpoint
checkpoint_name = f'ensemble_dirichlet_only_alpha{CONFIG["dirichlet_alpha"]}_checkpoint.pth'
torch.save({
    'training_rounds': training_rounds,
    'model_state_dict': ensemble_model.state_dict(),
    'cluster_labels': cluster_labels,
    'best_val_acc': max(val_accs),
    'final_test_acc': final_test_acc,
    'config': CONFIG,
    'cluster_val_accs': cluster_val_accs  # NEW: Save metric history in checkpoint too
}, checkpoint_name)

print(f"Model checkpoint saved to '{checkpoint_name}'")


## Summary

**Ensemble Method with Label Heterogeneity Only:**

**Key Characteristics:**
- **NO rotation-based feature heterogeneity**: All clients use same preprocessing
- **ONLY label heterogeneity**: Dirichlet distribution creates non-IID label distributions
- **Pooled training**: Data within clusters is combined (not federated)

**Research Question:**
Can hierarchical clustering capture meaningful patterns based on label distributions alone, without feature heterogeneity cues?

**Expected Findings:**
- Clustering quality may be lower without rotation signals
- Silhouette score indicates how well label patterns drive clustering
- Performance vs alpha shows ensemble's ability to handle pure label heterogeneity

**Comparison Goals:**
- vs **Ensemble (Rotation + Dirichlet)**: Impact of dual heterogeneity
- vs **FedAvg (Dirichlet Only)**: Hierarchical vs flat architecture for label heterogeneity
- Across alpha values: How ensemble handles varying degrees of label imbalance